# COMP5339 Assignment 2 - Data Retrieval and Caching

This notebook retrieves OpenElectricity data for the NEM network from 2026-05-01 00:00 to 2026-05-08 00:00 using 5-minute intervals. It caches the API responses and creates CSV files for the later MQTT and dashboard steps.

## 1. Setup

In [ ]:
# Libraries for API requests, JSON cache files, dates, file paths, and table processing.
import json
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests


# Use the notebook folder as the base folder for all generated files.
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
CACHE_DIR = OUTPUT_DIR / "cache"
PROCESSED_DIR = OUTPUT_DIR / "processed"

# Create output folders if they do not already exist.
for directory in [OUTPUT_DIR, CACHE_DIR, PROCESSED_DIR]:
    directory.mkdir(exist_ok=True)

# data scope: NEM, one full week in May 2026, from 2026-05-01 to 2026-05-08 with 5-minute intervals.
BASE_URL = "https://api.openelectricity.org.au/v4"
NETWORK_CODE = "NEM"
DATE_START = "2026-05-01T00:00:00"
DATE_END = "2026-05-08T00:00:00"
INTERVAL = "5m"

# Required facility metrics, and optional market metrics for price and demand.
FACILITY_METRICS = ["power", "emissions"]
MARKET_METRICS = ["price", "demand"]

# Batch facility requests to stay within API limits; reuse cache by default.
FACILITY_BATCH_SIZE = 25
REQUEST_TIMEOUT_SECONDS = 300
REFRESH_API = False

# Raw API responses are cached as JSON so the notebook can be rerun safely.
FACILITIES_CACHE = CACHE_DIR / "openelectricity_facilities_nem.json"
FACILITY_DATA_CACHE = CACHE_DIR / "openelectricity_facility_data_batched_nem_2026-05-01_2026-05-08.json"
MARKET_DATA_CACHE = CACHE_DIR / "openelectricity_market_price_demand_nem_2026-05-01_2026-05-08.json"

# Final cleaned CSV outputs.
FACILITY_CSV = PROCESSED_DIR / "consolidated_facility_power_emissions.csv"
MARKET_CSV = PROCESSED_DIR / "market_price_demand.csv"


## 2. API Key

In [ ]:
import os

OPENELECTRICITY_API_KEY = os.environ.get("OPENELECTRICITY_API_KEY")
if not OPENELECTRICITY_API_KEY:
    raise RuntimeError("Set OPENELECTRICITY_API_KEY in your environment before running retrieval.")
HEADERS = {
    "Authorization": f"Bearer {OPENELECTRICITY_API_KEY}",
    "Accept": "application/json",
}


## 3. Helper Functions

In [ ]:
def get_json(cache_path, endpoint, params=None, refresh=False):
    # Reuse the cached JSON file unless a fresh API request is required.
    if cache_path.exists() and not refresh:
        return json.loads(cache_path.read_text(encoding="utf-8"))

    # Send a GET request to the selected OpenElectricity API endpoint.
    response = requests.get(
        f"{BASE_URL}{endpoint}",
        headers = HEADERS,
        params = params,
        timeout = REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()

    # Stop if the API returns an unsuccessful response.
    if payload.get("success") is False:
        raise RuntimeError(payload.get("error", "OpenElectricity API request failed."))

    # Save the raw API response as JSON so later runs can use the cache.
    cache_path.write_text(json.dumps(payload, indent = 2), encoding = "utf-8")
    return payload


def flatten_points(points):
    # Convert OpenElectricity time-series points into a consistent list of dictionaries.
    rows = []
    if not isinstance(points, list):
        return rows

    for point in points:
        if isinstance(point, dict):
            rows.append(point)
        elif isinstance(point, list) and len(point) >= 2:
            rows.append({"timestamp": point[0], "value": point[1]})

    return rows


## 4. Retrieve Facility Metadata

In [ ]:
# Retrieve facility metadata for the selected network.
facilities_payload = get_json(
    FACILITIES_CACHE,
    "/facilities/",
    {"network_id": NETWORK_CODE},
    refresh = REFRESH_API,
)

facility_rows = []
unit_to_facility = {}

for facility in facilities_payload.get("data", []):
    # Keep only facilities from the selected network.
    if facility.get("network_id") != NETWORK_CODE:
        continue

    facility_code = str(facility.get("code", "")).strip()
    if not facility_code:
        continue

    units = facility.get("units") or []
    fueltechs = sorted({unit.get("fueltech_id") for unit in units if unit.get("fueltech_id")})

    # Store one metadata row for each facility.
    facility_rows.append(
        {
            "facility_code": facility_code,
            "facility_name": facility.get("name"),
            "network_id": facility.get("network_id"),
            "network_region": facility.get("network_region"),
            "fueltech": ", ".join(fueltechs),
            "metadata_unit_count": len(units),
        }
    )

    # Map the facility code and all unit codes back to the parent facility.
    unit_to_facility[facility_code] = facility_code
    for unit in units:
        unit_code = unit.get("code")
        if unit_code:
            unit_to_facility[str(unit_code).strip()] = facility_code

# These load-side unit names appear in the API time series but map to these facilities.
unit_to_facility.update({"BBATRYL1": "BBATTERY", "DALNTH01": "DALNTH"})

facilities_df = pd.DataFrame(facility_rows).drop_duplicates("facility_code")
facility_codes = sorted(facilities_df["facility_code"].dropna().astype(str).unique())

print("NEM facilities in metadata:", len(facilities_df))
print("Facility codes used for API requests:", len(facility_codes))
facilities_df.head()


## 5. Retrieve Power and Emissions Data

In [ ]:
# Split the selected week into daily chunks.
start = datetime.fromisoformat(DATE_START)
end = datetime.fromisoformat(DATE_END)

date_ranges = []
current = start
while current < end:
    next_end = min(current + timedelta(days=1), end)
    date_ranges.append((current.isoformat(), next_end.isoformat()))
    current = next_end

# Split facilities into smaller batches to keep each API request manageable.
facility_batches = [
    facility_codes[i : i + FACILITY_BATCH_SIZE]
    for i in range(0, len(facility_codes), FACILITY_BATCH_SIZE)
]

print("Date chunks:", len(date_ranges))
print("Facility batches:", len(facility_batches))
print("Planned facility data requests:", len(date_ranges) * len(facility_batches))

# Use the combined cache if it already exists.
if FACILITY_DATA_CACHE.exists() and not REFRESH_API:
    facility_payload = json.loads(FACILITY_DATA_CACHE.read_text(encoding = "utf-8"))
else:
    # Otherwise, download each date chunk and facility batch from the API.
    facility_payload = {
        "success": True,
        "data": [],
        "request_plan": {
            "facility_count": len(facility_codes),
            "facility_batch_size": FACILITY_BATCH_SIZE,
            "facility_batches": len(facility_batches),
            "date_chunks": len(date_ranges),
        },
    }

    for chunk_start, chunk_end in date_ranges:
        for batch_number, batch_codes in enumerate(facility_batches, start=1):
            chunk_name = (
                f"openelectricity_facility_data_{NETWORK_CODE}_{chunk_start[:10]}_"
                f"{chunk_end[:10]}_batch_{batch_number:03d}.json"
            )
            chunk_path = CACHE_DIR / chunk_name
            params = {
                "metrics": FACILITY_METRICS,
                "interval": INTERVAL,
                "date_start": chunk_start,
                "date_end": chunk_end,
                "facility_code": batch_codes,
            }
            payload = get_json(
                chunk_path,
                f"/data/facilities/{NETWORK_CODE}",
                params,
                refresh = REFRESH_API,
            )
            facility_payload["data"].extend(payload.get("data", []))

    # Save all downloaded chunks into one combined cache file.
    FACILITY_DATA_CACHE.write_text(json.dumps(facility_payload, indent = 2), encoding = "utf-8")

print("Facility API data blocks:", len(facility_payload.get("data", [])))


## 6. Clean Facility Time Series

In [ ]:
facility_records = []

# Loop through each API data block, such as power or emissions.
for series in facility_payload.get("data", []):
    if not isinstance(series, dict):
        continue

    metric = str(series.get("metric", "")).lower().strip()
    if metric not in FACILITY_METRICS:
        continue

    # Each result usually represents one unit's time series.
    for result in series.get("results", []):
        if not isinstance(result, dict):
            continue

        columns = result.get("columns") or {}
        unit_code = columns.get("unit_code") or columns.get("facility_code")

        # If unit_code is not in columns, try to extract it from the result name.
        if not unit_code and isinstance(result.get("name"), str):
            prefix = f"{metric}_"
            if result["name"].startswith(prefix):
                unit_code = result["name"][len(prefix) :]

        # Convert each timestamp-value point into one row.
        for point in flatten_points(result.get("data")):
            facility_records.append(
                {
                    "timestamp": point.get("timestamp"),
                    "unit_code": unit_code,
                    "facility_code": unit_to_facility.get(str(unit_code).strip()) if unit_code else None,
                    "metric": metric,
                    "value": point.get("value"),
                }
            )

timeseries_df = pd.DataFrame(facility_records)

# Standardise data types before cleaning and aggregation.
timeseries_df["timestamp"] = pd.to_datetime(timeseries_df["timestamp"], errors = "coerce", utc = True)
timeseries_df["unit_code"] = timeseries_df["unit_code"].astype("string").str.strip()
timeseries_df["facility_code"] = timeseries_df["facility_code"].astype("string").str.strip()
timeseries_df["metric"] = timeseries_df["metric"].astype(str).str.lower().str.strip()
timeseries_df["value"] = pd.to_numeric(timeseries_df["value"], errors = "coerce")

# Remove invalid and duplicated records.
timeseries_df = timeseries_df.dropna(
    subset = ["timestamp", "unit_code", "facility_code", "metric", "value"]
)
timeseries_df = timeseries_df.drop_duplicates(
    subset = ["timestamp", "unit_code", "facility_code", "metric"]
)

print("Clean facility time-series rows:", len(timeseries_df))
timeseries_df.head()


## 7. Aggregate to Facility Level and Save CSV

In [ ]:
# Pivot the long table so each unit has power_mw and emissions_tonnes columns.
unit_level = (
    timeseries_df.pivot_table(
        index = ["timestamp", "facility_code", "unit_code"],
        columns = "metric",
        values = "value",
        aggfunc = "sum",
    )
    .reset_index()
    .rename(columns = {"power": "power_mw", "emissions": "emissions_tonnes"})
)

# Ensure both required numeric columns exist before aggregation.
for col in ["power_mw", "emissions_tonnes"]:
    if col not in unit_level.columns:
        unit_level[col] = 0.0
    unit_level[col] = pd.to_numeric(unit_level[col], errors = "coerce").fillna(0.0)

# Keep signed power values, but prevent negative emissions in the cleaned output.
unit_level.loc[unit_level["emissions_tonnes"] < 0, "emissions_tonnes"] = 0.0

# Attach facility metadata such as name, region, and fuel technology.
facility_with_meta = unit_level.merge(facilities_df, on = "facility_code", how = "left")

# Aggregate unit-level rows to facility-level rows for each timestamp.
final_df = (
    facility_with_meta.groupby(["timestamp", "facility_code"], as_index=False)
    .agg(
        facility_name = ("facility_name", "first"),
        network_id = ("network_id", "first"),
        network_region = ("network_region", "first"),
        fueltech = ("fueltech", "first"),
        unit_count = ("unit_code", "nunique"),
        power_mw = ("power_mw", "sum"),
        emissions_tonnes = ("emissions_tonnes", "sum"),
    )
    .sort_values(["timestamp", "facility_code"])
    .reset_index(drop = True)
)

# Format timestamps and keep the final CSV column order.
final_df["timestamp"] = final_df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
final_df = final_df[
    [
        "timestamp",
        "facility_code",
        "facility_name",
        "network_id",
        "network_region",
        "fueltech",
        "unit_count",
        "power_mw",
        "emissions_tonnes",
    ]
]

# Save the cleaned facility-level dataset.
final_df.to_csv(FACILITY_CSV, index = False)

print("Facility CSV saved to:", FACILITY_CSV.relative_to(BASE_DIR))
print("Rows:", len(final_df))
print("Facilities:", final_df["facility_code"].nunique())
final_df.head()


## 8. Retrieve Optional Market Price and Demand Data

In [ ]:
# Retrieve optional market price and demand data for the same network, date range, and interval.
market_payload = get_json(
    MARKET_DATA_CACHE,
    f"/market/network/{NETWORK_CODE}",
    {
        "metrics": MARKET_METRICS,
        "interval": INTERVAL,
        "date_start": DATE_START,
        "date_end": DATE_END,
    },
    refresh = REFRESH_API,
)

market_records = []

# Flatten market API blocks into one timestamp-market-series-metric-value table.
for series in market_payload.get("data", []):
    if not isinstance(series, dict):
        continue

    metric = str(series.get("metric", "")).lower().strip()
    if metric not in MARKET_METRICS:
        continue

    # Each market result is a region-level or aggregate market time series.
    for result in series.get("results", []):
        if not isinstance(result, dict):
            continue

        columns = result.get("columns") or {}
        region = columns.get("region") or columns.get("network_region")

        # The market series label can come from columns, or sometimes from the result name.
        if not region and isinstance(result.get("name"), str) and "_" in result["name"]:
            region = result["name"].split("_", 1)[1]

        # Convert each timestamp-value point into one row.
        for point in flatten_points(result.get("data")):
            market_records.append(
                {
                    "timestamp": point.get("timestamp"),
                    "network_region": region,
                    "metric": metric,
                    "value": point.get("value"),
                }
            )

market_long = pd.DataFrame(market_records)

# Standardise market data types before pivoting.
market_long["timestamp"] = pd.to_datetime(market_long["timestamp"], errors = "coerce", utc = True)
market_long["network_region"] = market_long["network_region"].astype("string").str.strip()
market_long["metric"] = market_long["metric"].astype(str).str.lower().str.strip()
market_long["value"] = pd.to_numeric(market_long["value"], errors = "coerce")

# Remove invalid or duplicate market records.
market_long = market_long.dropna(subset=["timestamp", "network_region", "metric", "value"])
market_long = market_long.drop_duplicates(subset=["timestamp", "network_region", "metric"])

# Pivot price and demand into separate columns for each timestamp and market series.
market_df = (
    market_long.pivot_table(
        index = ["timestamp", "network_region"],
        columns = "metric",
        values = "value",
        aggfunc = "first",
    )
    .reset_index()
    .rename(columns = {"price": "price_aud_per_mwh", "demand": "demand_mw"})
)

# Ensure both optional market columns exist and are numeric.
for col in ["price_aud_per_mwh", "demand_mw"]:
    if col not in market_df.columns:
        market_df[col] = pd.NA
    market_df[col] = pd.to_numeric(market_df[col], errors = "coerce")

# Sort, format timestamps, and save the optional market CSV.
market_df = market_df.sort_values(["timestamp", "network_region"]).reset_index(drop = True)
market_df["timestamp"] = market_df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
market_df = market_df[["timestamp", "network_region", "price_aud_per_mwh", "demand_mw"]]

market_df.to_csv(MARKET_CSV, index=False)

print("Market CSV saved to:", MARKET_CSV.relative_to(BASE_DIR))
print("Rows:", len(market_df))
print("Market series:", market_df["network_region"].nunique())
market_df.head()


## 9. Validation Summary

In [ ]:
# Read the saved CSV files back from disk to confirm they are valid.
check_df = pd.read_csv(FACILITY_CSV)
market_check_df = pd.read_csv(MARKET_CSV)

# Facility CSV validation: row counts, coverage, duplicates, and missing values.
print("Facility CSV Validation Summary")
print("Rows:", len(check_df))
print("Facilities:", check_df["facility_code"].nunique())
print("Unique timestamps:", check_df["timestamp"].nunique())
print("Timestamp range:", check_df["timestamp"].min(), "to", check_df["timestamp"].max())
print("Duplicate timestamp + facility rows:", check_df.duplicated(["timestamp", "facility_code"]).sum())
print("Missing power_mw:", check_df["power_mw"].isna().sum())
print("Missing emissions_tonnes:", check_df["emissions_tonnes"].isna().sum())
print("Negative emissions rows:", int((check_df["emissions_tonnes"] < 0).sum()))
print("Network IDs:", sorted(check_df["network_id"].dropna().unique()))
print("Network regions:", sorted(check_df["network_region"].dropna().unique()))

# Market CSV validation: row counts, market-series coverage, duplicates, and missing values.
print("\nMarket CSV Validation Summary")
print("Rows:", len(market_check_df))
print("Market series:", market_check_df["network_region"].nunique())
print("Unique timestamps:", market_check_df["timestamp"].nunique())
print("Timestamp range:", market_check_df["timestamp"].min(), "to", market_check_df["timestamp"].max())
print("Duplicate timestamp + market-series rows:", market_check_df.duplicated(["timestamp", "network_region"]).sum())
print("Missing price:", market_check_df["price_aud_per_mwh"].isna().sum())
print("Missing demand:", market_check_df["demand_mw"].isna().sum())


## 10. Task 4 - Assignment 1 Schema in DuckDB

The assignment also asks us to implement the schema from Assignment 1 in a database. In Assignment 1, we used DuckDB and had a CER power station table as the main facility table. Here I create a local DuckDB database again, load the Assignment 1 facility/location data, and then add the Assignment 2 power/emissions stream tables.

This database is useful because the dashboard can be explained as joining the live MQTT facility values with the Assignment 1 facility/location table.

In [ ]:
# DuckDB is used 
import duckdb
import json
import math
import pandas as pd
import difflib
import hashlib
import re
from pathlib import Path
from IPython.display import display

DB_PATH = Path("assignment2_electricity.duckdb")
ASSIGNMENT1_CER_CSV = Path("assignment1/cer_augmented.csv")
FACILITY_STREAM_CSV = Path("outputs/processed/consolidated_facility_power_emissions.csv")
MARKET_STREAM_CSV = Path("outputs/processed/market_price_demand.csv")
OE_FACILITIES_CACHE = Path("outputs/cache/openelectricity_facilities_nem.json")

print("Assignment 1 CSV exists:", ASSIGNMENT1_CER_CSV.exists())
print("Assignment 2 facility CSV exists:", FACILITY_STREAM_CSV.exists())
print("Assignment 2 market CSV exists:", MARKET_STREAM_CSV.exists())
print("OpenElectricity facility metadata cache exists:", OE_FACILITIES_CACHE.exists())


In [ ]:
REGION_FALLBACK = {
    "NSW1": (-32.2, 147.0),
    "QLD1": (-22.5, 144.5),
    "VIC1": (-36.8, 144.6),
    "SA1": (-31.9, 136.6),
    "TAS1": (-42.0, 146.6),
}


def normalise_name(value):
    # Make names easier to match between Ass 1 and OpenElectricity.
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    stop_words = {"power", "station", "farm", "solar", "wind", "battery", "pty", "ltd", "the", "and", "w", "sgu"}
    return " ".join(word for word in text.split() if word not in stop_words)


def fallback_location(region, facility_code):
    # If no real coordinate is available, place the marker near the NEM region.
    lat, lon = REGION_FALLBACK.get(str(region), (-25.5, 134.0))
    digest = hashlib.md5(str(facility_code).encode("utf-8")).hexdigest()
    angle = int(digest[:4], 16) / 65535 * 2 * math.pi
    radius = 0.35 + (int(digest[4:8], 16) / 65535) * 1.2
    return lat + math.sin(angle) * radius, lon + math.cos(angle) * radius


def load_openelectricity_locations(cache_path=OE_FACILITIES_CACHE):
    # OpenElectricity locations may be used to enrich Ass 1 locations.
    # This helper reads the cached facility metadata and keeps only code + lat/lng.
    columns = ["facility_code", "oe_facility_name", "oe_latitude", "oe_longitude"]
    if not cache_path.exists():
        return pd.DataFrame(columns=columns)

    payload = json.loads(cache_path.read_text(encoding="utf-8"))
    facilities = payload.get("data", []) if isinstance(payload, dict) else payload

    rows = []
    for item in facilities:
        if not isinstance(item, dict):
            continue
        location = item.get("location") or {}
        lat = location.get("lat", location.get("latitude"))
        lon = location.get("lng", location.get("lon", location.get("longitude")))
        if lat is None or lon is None:
            continue
        rows.append({
            "facility_code": str(item.get("code", "")).strip(),
            "oe_facility_name": str(item.get("name", "")).strip(),
            "oe_latitude": float(lat),
            "oe_longitude": float(lon),
        })

    result = pd.DataFrame(rows, columns=columns)
    if not result.empty:
        result = result.drop_duplicates("facility_code")
    return result


def build_location_match_table(facility_df, ass1_df, oe_locations=None):
    # Start with one row per OpenElectricity facility.
    locations = facility_df[["facility_code", "facility_name", "network_region", "fueltech"]].drop_duplicates("facility_code").copy()
    locations["latitude"] = pd.NA
    locations["longitude"] = pd.NA
    locations["assignment1_match_name"] = ""
    locations["location_source"] = "region_fallback"

    # First try Ass1 names
    ass1 = ass1_df.dropna(subset=["facility_name", "latitude", "longitude"]).copy()
    ass1["match_key"] = ass1["facility_name"].map(normalise_name)
    key_to_row = {}
    for _, row in ass1.iterrows():
        key = row["match_key"]
        if key and key not in key_to_row:
            key_to_row[key] = row

    keys = list(key_to_row.keys())

    for idx, row in locations.iterrows():
        key = normalise_name(row["facility_name"])
        match_row = None
        match_type = ""

        if key in key_to_row:
            match_row = key_to_row[key]
            match_type = "assignment1_exact"
        else:
            for ass1_key in keys:
                if len(key) >= 4 and len(ass1_key) >= 4 and (key in ass1_key or ass1_key in key):
                    match_row = key_to_row[ass1_key]
                    match_type = "assignment1_contains"
                    break

        if match_row is None and key:
            close = difflib.get_close_matches(key, keys, n=1, cutoff=0.82)
            if close:
                match_row = key_to_row[close[0]]
                match_type = "assignment1_fuzzy"

        if match_row is not None:
            locations.loc[idx, "latitude"] = float(match_row["latitude"])
            locations.loc[idx, "longitude"] = float(match_row["longitude"])
            locations.loc[idx, "assignment1_match_name"] = str(match_row["facility_name"])
            locations.loc[idx, "location_source"] = match_type

    #  use OpenElectricity official facility metadata to supplement missing Ass 1 matches.
    
    if oe_locations is not None and not oe_locations.empty:
        oe_by_code = oe_locations.set_index("facility_code").to_dict("index")
        for idx, row in locations.iterrows():
            if pd.notna(row["latitude"]) and pd.notna(row["longitude"]):
                continue
            oe_row = oe_by_code.get(str(row["facility_code"]))
            if oe_row is None:
                continue
            locations.loc[idx, "latitude"] = oe_row["oe_latitude"]
            locations.loc[idx, "longitude"] = oe_row["oe_longitude"]
            locations.loc[idx, "assignment1_match_name"] = oe_row["oe_facility_name"]
            locations.loc[idx, "location_source"] = "openelectricity_metadata"

    for idx, row in locations.iterrows():
        if pd.isna(row["latitude"]) or pd.isna(row["longitude"]):
            lat, lon = fallback_location(row["network_region"], row["facility_code"])
            locations.loc[idx, "latitude"] = lat
            locations.loc[idx, "longitude"] = lon

    locations["latitude"] = pd.to_numeric(locations["latitude"], errors="coerce")
    locations["longitude"] = pd.to_numeric(locations["longitude"], errors="coerce")
    return locations


In [ ]:
# Read Assignment 1 CSV if available; otherwise continue with an empty table.
if ASSIGNMENT1_CER_CSV.exists():
    ass1_cer = pd.read_csv(ASSIGNMENT1_CER_CSV)
    print("Loaded Assignment 1 CER CSV:", ASSIGNMENT1_CER_CSV)
else:
    print("Assignment 1 CER CSV not found. Continuing with OpenElectricity metadata and regional fallback locations.")
    ass1_cer = pd.DataFrame(columns=["facility_name", "latitude", "longitude", "state", "fuel_source"])

# Read Ass 2 CSV files.
facility_stream = pd.read_csv(FACILITY_STREAM_CSV)
market_stream = pd.read_csv(MARKET_STREAM_CSV)
oe_locations = load_openelectricity_locations()

# Make timestamp columns proper datetimes before storing them.
facility_stream["timestamp"] = pd.to_datetime(facility_stream["timestamp"], errors="coerce", utc=True)
market_stream["timestamp"] = pd.to_datetime(market_stream["timestamp"], errors="coerce", utc=True)

facility_locations = build_location_match_table(facility_stream, ass1_cer, oe_locations)

# Create a local DuckDB database file.
con = duckdb.connect(str(DB_PATH))

# Ass 1 schema: CER station
con.register("ass1_cer_df", ass1_cer)
con.execute("CREATE OR REPLACE TABLE cer_stations AS SELECT * FROM ass1_cer_df")
con.unregister("ass1_cer_df")

# Ass 2 tables: stream data and optional market data.
con.register("facility_stream_df", facility_stream)
con.execute("CREATE OR REPLACE TABLE facility_power_emissions AS SELECT * FROM facility_stream_df")
con.unregister("facility_stream_df")

con.register("market_stream_df", market_stream)
con.execute("CREATE OR REPLACE TABLE market_price_demand AS SELECT * FROM market_stream_df")
con.unregister("market_stream_df")

# Matching table between Ass 2 facilities and Ass 1 / OpenElectricity locations.
con.register("facility_locations_df", facility_locations)
con.execute("CREATE OR REPLACE TABLE facility_locations AS SELECT * FROM facility_locations_df")
con.unregister("facility_locations_df")

#  power/emissions already joined with locations.
con.execute("""
CREATE OR REPLACE TABLE facility_power_emissions_integrated AS
SELECT
    f.*,
    l.latitude,
    l.longitude,
    l.assignment1_match_name,
    l.location_source
FROM facility_power_emissions f
LEFT JOIN facility_locations l USING (facility_code, facility_name, network_region, fueltech)
ORDER BY f.timestamp, f.facility_code
""")

# Latest facility table for dashboard queries.
con.execute("""
CREATE OR REPLACE TABLE latest_facility_status AS
WITH ranked AS (
    SELECT
        f.*,
        row_number() OVER (PARTITION BY facility_code ORDER BY timestamp DESC) AS rn
    FROM facility_power_emissions_integrated f
)
SELECT
    timestamp,
    facility_code,
    facility_name,
    network_id,
    network_region,
    fueltech,
    unit_count,
    power_mw,
    emissions_tonnes,
    latitude,
    longitude,
    assignment1_match_name,
    location_source
FROM ranked
WHERE rn = 1
""")

print("Database saved to:", DB_PATH)
print("Tables in database:")
display(con.execute("SHOW TABLES").fetchdf())

for table in [
    "cer_stations",
    "facility_power_emissions",
    "facility_power_emissions_integrated",
    "market_price_demand",
    "facility_locations",
    "latest_facility_status",
]:
    count = con.execute(f"SELECT COUNT(*) AS rows FROM {table}").fetchdf()
    print(table, int(count.loc[0, "rows"]))

print("Location sources:")
display(con.execute("SELECT location_source, COUNT(*) AS facilities FROM facility_locations GROUP BY location_source ORDER BY facilities DESC").fetchdf())

con.close()


In [ ]:
# Small check query for the report: latest power/emissions with location.
con = duckdb.connect(str(DB_PATH))
query = """
SELECT
    facility_code,
    facility_name,
    network_region,
    fueltech,
    round(power_mw, 2) AS power_mw,
    round(emissions_tonnes, 3) AS emissions_tonnes,
    latitude,
    longitude,
    location_source
FROM latest_facility_status
ORDER BY power_mw DESC
LIMIT 10
"""
display(con.execute(query).fetchdf())
con.close()

## 10. Task 3 - MQTT Publishing

This section uses the cleaned CSV from Task 2 and publishes the facility power + emissions records as MQTT messages.The process is: read data, make JSON, publish, then wait a little before the next message.

The publisher sends already-integrated records. Each MQTT message includes the facility power/emissions values plus the matched latitude, longitude and location source from the Task 4 database.

The publisher sends records in event-time order. For continuous execution it publishes a chunk of rows each round, waits 0.1 seconds between messages, then waits 60 seconds before the next round. It also skips a facility row if the power and emissions are exactly the same as the last message for that facility, so we only publish updated values.


In [ ]:
# Libraries used for the MQTT publishing task.
import time
import math
import json
import socket
import pandas as pd
import duckdb
from pathlib import Path


import paho.mqtt.client as mqtt

# MQTT settings.


MQTT_BROKER = "localhost"
MQTT_PORT = 1883
MQTT_TOPIC = "COMP5339/assignment2/power_emissions"
MQTT_CLIENT_ID = "comp5339_assignment2_publisher"
MQTT_CONNECT_TIMEOUT_SECONDS = 8


PUBLISH_DELAY_SECONDS = 0.1
REPLAY_RESTART_DELAY_SECONDS = 60
PUBLISH_CHUNK_SIZE = 3000

# If this notebook is copied into the final folder, these paths still work.
FACILITY_CSV = Path("outputs/processed/consolidated_facility_power_emissions.csv")
MARKET_CSV = Path("outputs/processed/market_price_demand.csv")
DB_PATH = Path("assignment2_electricity.duckdb")

print("Facility CSV exists:", FACILITY_CSV.exists())
print("Market CSV exists:", MARKET_CSV.exists())
print("Task 4 DuckDB exists:", DB_PATH.exists())


In [ ]:
def read_publish_data():
    # Read the facility CSV 
    facility_df = pd.read_csv(FACILITY_CSV)

    # Timestamp is converted so sorting is real chronological order.
    facility_df["event_time"] = pd.to_datetime(facility_df["timestamp"], errors="coerce", utc=True)
    facility_df = facility_df.dropna(subset=["event_time", "facility_code", "facility_name"])

    # Convert important values to numbers. Bad cells become 0 so publishing does not crash.
    for col in ["unit_count", "power_mw", "emissions_tonnes"]:
        facility_df[col] = pd.to_numeric(facility_df[col], errors="coerce").fillna(0)

    # Join Task 4 locations before MQTT publishing. 
    if DB_PATH.exists():
        con = duckdb.connect(str(DB_PATH), read_only=True)
        locations = con.execute("""
            SELECT facility_code, latitude, longitude, assignment1_match_name, location_source
            FROM facility_locations
        """).fetchdf()
        con.close()

        facility_df = facility_df.merge(locations, on="facility_code", how="left")
    else:
        # The notebook should normally create the DB first, but this keeps dry-run readable.
        facility_df["latitude"] = pd.NA
        facility_df["longitude"] = pd.NA
        facility_df["assignment1_match_name"] = ""
        facility_df["location_source"] = "missing_task4_db"

    # Event-time order first, then facility code to keep a stable order inside the same 5-minute timestamp.
    facility_df = facility_df.sort_values(["event_time", "facility_code"]).reset_index(drop=True)

    # Optional price and demand data. It is matched by timestamp.
    market_by_time = {}
    if MARKET_CSV.exists():
        market_df = pd.read_csv(MARKET_CSV)
        market_df["event_time"] = pd.to_datetime(market_df["timestamp"], errors="coerce", utc=True)
        market_df = market_df.dropna(subset=["event_time"])
        market_df = market_df.drop_duplicates("event_time")
        market_by_time = market_df.set_index("event_time").to_dict("index")

    return facility_df, market_by_time


def safe_number(value):
    # JSON does not like NaN or infinity, so this small helper cleans them.
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    if isinstance(value, float) and not math.isfinite(value):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return value
    if not math.isfinite(number):
        return None
    return round(number, 6)


def safe_text(value):
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    return str(value)


def make_mqtt_message(row, market_by_time, replay_number, event_number):
    # Combine power, emissions, location, and optional market data into one message.
    market = market_by_time.get(row["event_time"], {})

    return {
        "timestamp": row["timestamp"],
        "facility_code": str(row["facility_code"]),
        "facility_name": str(row["facility_name"]),
        "network_id": str(row.get("network_id", "")),
        "network_region": str(row.get("network_region", "")),
        "fueltech": str(row.get("fueltech", "")),
        "unit_count": int(row.get("unit_count", 0)),
        "power_mw": safe_number(row.get("power_mw", 0)),
        "emissions_tonnes": safe_number(row.get("emissions_tonnes", 0)),
        "latitude": safe_number(row.get("latitude")),
        "longitude": safe_number(row.get("longitude")),
        "assignment1_match_name": safe_text(row.get("assignment1_match_name", "")),
        "location_source": safe_text(row.get("location_source", "")),
        "price_aud_per_mwh": safe_number(market.get("price_aud_per_mwh")),
        "demand_mw": safe_number(market.get("demand_mw")),
        "source": "Task 2 CSV joined with Task 4 Assignment 1 location table",
        "replay_number": replay_number,
        "event_number": event_number,
    }


In [ ]:
def publish_messages(loop_forever=True, dry_run=False, max_messages=0, chunk_size=PUBLISH_CHUNK_SIZE):
    # Read once from the cached CSV. This avoids repeated API calls during streaming.
    facility_df, market_by_time = read_publish_data()
    total_rows = len(facility_df)
    print("Rows available:", total_rows)
    print("Facilities:", facility_df["facility_code"].nunique())
    print("Rows with coordinates:", facility_df[["latitude", "longitude"]].dropna().shape[0])

    # Connect only when this is not a dry run.
    client = None
    if not dry_run:
        client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=MQTT_CLIENT_ID)
        old_timeout = socket.getdefaulttimeout()
        socket.setdefaulttimeout(MQTT_CONNECT_TIMEOUT_SECONDS)
        try:
            client.connect(MQTT_BROKER, MQTT_PORT, 60)
        except Exception as exc:
            print("MQTT connection failed:", exc)
            print("Check that local Mosquitto is running, or change MQTT_BROKER if using a remote broker.")
            return
        finally:
            socket.setdefaulttimeout(old_timeout)

        client.loop_start()
        print("Connected to MQTT broker:", MQTT_BROKER, MQTT_PORT)

    replay_number = 1
    start_index = 0
    total_published = 0
    last_values = {}

    try:
        while True:
            if total_rows == 0:
                print("No rows to publish.")
                break

            end_index = min(start_index + chunk_size, total_rows)
            chunk = facility_df.iloc[start_index:end_index]
            print(f"Round {replay_number}: rows {start_index + 1}-{end_index} of {total_rows}")

            published_this_round = 0

            for _, row in chunk.iterrows():
                facility_code = str(row["facility_code"])
                current_values = (
                    round(float(row["power_mw"]), 6),
                    round(float(row["emissions_tonnes"]), 6),
                )

                #  only publish updated power/emissions values.
                if last_values.get(facility_code) == current_values:
                    continue

                last_values[facility_code] = current_values
                total_published = total_published + 1
                published_this_round = published_this_round + 1

                message = make_mqtt_message(row, market_by_time, replay_number, total_published)
                payload = json.dumps(message, ensure_ascii=False)

                if dry_run:
                    print(payload)
                else:
                    info = client.publish(MQTT_TOPIC, payload=payload, qos=1, retain=False)
                    info.wait_for_publish()

                if max_messages and total_published >= max_messages:
                    print("Stopped early because max_messages was reached.")
                    return

                #  0.1 second delay between published messages.
                time.sleep(PUBLISH_DELAY_SECONDS)

            print("Round finished. New messages published:", published_this_round)

            if not loop_forever:
                break

            start_index = end_index
            if start_index >= total_rows:
                start_index = 0
                print("Reached the end of the CSV, so the next round starts again from the beginning.")

            # each continuous round should wait 60 seconds before the next round.
            print("Waiting 60 seconds before the next round...")
            time.sleep(REPLAY_RESTART_DELAY_SECONDS)
            replay_number = replay_number + 1

    finally:
        if client is not None:
            client.loop_stop()
            client.disconnect()
            print("MQTT disconnected")


In [ ]:
# Quick check: print a few messages without using the MQTT broker.
# The printed JSON  include latitude and longitude.
publish_messages(loop_forever=False, dry_run=True, max_messages=5, chunk_size=20)


In [ ]:

# Keep this False while testing the notebook, because the real publisher runs continuously.
RUN_REAL_MQTT_PUBLISHER = True

if RUN_REAL_MQTT_PUBLISHER:
    publish_messages(loop_forever=True, dry_run=False, chunk_size=PUBLISH_CHUNK_SIZE)
else:
    print("Real MQTT publisher is not running. Change RUN_REAL_MQTT_PUBLISHER to True for the demo.")
